# Tarea 2 · Parte 2 — Evaluación, comparación de modelos y extracción estructurada

**Pregunta guía:** ¿Pueden convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

Este notebook usa **solo** los archivos recolectados en la Parte 1 (`01_extraccion.ipynb`):

| Paso | Qué se hace | Modelos |
|---|---|---|
| 1 | Muestra fija de **10 documentos** y **10 clips** con transcripción manual (`texto_referencia`) | — |
| 2 | Línea base | PaddleOCR 3.7 base · faster-whisper `small` (CPU, int8) |
| 3 | Modelos alternativos sobre **las mismas** muestras | Tesseract 5 (`spa`) · Whisper large-v3-turbo (Parte 1) · Qwen3-ASR 0.6B (Sesión 13) |
| 4 | Extracción estructurada con LLM + auditoría humana de 20 casos | Gemini (API) u Ollama local |
| 5 | Casos difíciles, tabla comparativa y decisión | — |

**Reglas que sigo:** las configuraciones de todos los modelos se fijan **antes** de ver las métricas; todos los modelos se evalúan sobre los mismos 10 + 10 archivos;
las referencias son transcripciones humanas (no salidas de otro modelo).

### Instalación (una vez, en la terminal, dentro de `tarea_2`)
```
.venv\Scripts\python.exe -m pip install pytesseract "transformers>=5.16" accelerate google-genai python-dotenv
winget install UB-Mannheim.TesseractOCR
```
Para el LLM: crear un archivo `tarea_2/.env` (ya está en `.gitignore`) con la línea `GEMINI_API_KEY=tu_clave` (clave gratis en aistudio.google.com).

In [3]:
import os, re, json, time, random, warnings, hashlib, platform
from datetime import date
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import torch                     # en Windows, torch antes que paddle
import numpy as np
import pandas as pd
import jiwer
import voz
from voz import SR, cargar_audio, normalizar

BASE = Path.cwd()
DATOS = BASE / "datos"
RAW = DATOS / "raw"
AUDIO = RAW / "audio"
REF = RAW / "referencias"; REF.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 90)

documentos = pd.read_csv(DATOS / "documentos.csv", dtype={"id": str}).fillna({"texto_ocr": "", "texto_referencia": ""})
transcripciones = pd.read_csv(DATOS / "transcripciones.csv", dtype={"id": str}).fillna({"texto_asr": "", "texto_referencia": ""})
ocr_crudo = pd.read_csv(RAW / "ocr_crudo.csv", dtype={"id": str}).fillna({"texto_ocr_crudo": ""})
inventario = pd.read_csv(AUDIO / "inventario_clips.csv", dtype={"id": str}).fillna({"texto_oficial": ""})
# Misma seudonimización de la Parte 1: se aplica a TODA salida de modelo antes de medir y de guardar
reemplazos = dict(pd.read_csv(RAW / "reemplazos_privados.csv", dtype=str, sep=None, engine="python",
                              encoding="utf-8-sig").dropna().values)
PRIVADOS = {"doc_014"}
def seud(texto, i=""):
    t = texto or ""
    for nombre, s in reemplazos.items():
        t = voz.seudonimizar(t, {nombre.strip(): s})
    t = re.sub(r"\bDNI\W{0,3}\d{8}\b", "[DNI]", t, flags=re.I)
    if i in PRIVADOS:
        t = re.sub(r"\b9\d{2}[\s-]?\d{3}[\s-]?\d{3}\b", "[TELEFONO]", t)
    return t

print("Python", platform.python_version(), "· torch", torch.__version__)
print(f"Documentos: {len(documentos)} · Clips: {len(transcripciones)} ({transcripciones.duracion_seg.sum() / 60:.1f} min)")

Python 3.12.15 · torch 2.14.1+cpu
Documentos: 25 · Clips: 24 (32.2 min)


## 1. Muestra de evaluación (fijada antes de medir)

La muestra se eligió **al azar con semilla fija (2026)** y quedó guardada con fecha en `modelos/config_evaluacion.json` **antes** de calcular cualquier métrica:
- **Documentos (10):** 3 impresos, 3 fotos y 4 manuscritos, sorteados entre los que tienen ≤ 1 500 caracteres de OCR
  (transcribir a mano un periódico entero no es viable). El criterio es el largo, no el error del OCR.
- **Clips (10):** las 3 entrevistas más cortas de Wikimedia Commons (calle, con ruido) + 7 clips de lectura de OpenSLR 73 sorteados.

### Cómo se hicieron las referencias manuales
- **Documentos:** transcribí cada imagen a mano en `datos/raw/referencias/doc_XXX.txt`, **copiando la ortografía original** (p. ej. "estranjero", "sufrajios")
  y escribiendo solo lo que se ve (sin completar palabras cortadas). En `doc_014` escribí `[PERSONA_1]` y `[TELEFONO]` en lugar de los datos del particular,
  porque la referencia se publica; por la misma razón, la salida de **todos** los modelos se seudonimiza igual antes de medir y guardar.
- **Entrevistas de Commons:** transcritas a mano escuchando el audio.
- **OpenSLR 73:** el dataset trae la transcripción humana del texto leído. La usé como punto de partida y **la verifiqué escuchando cada clip**, corrigiendo
  donde el lector se apartó del texto. No se usó ninguna salida de modelo como referencia.

In [4]:
# Muestra fijada antes de medir (modelos/config_evaluacion.json, semilla 2026)
cfg = json.loads((BASE / "modelos/config_evaluacion.json").read_text(encoding="utf-8"))
DOCS_EVAL_IDS = cfg["muestra_docs"]
CLIPS_EVAL_IDS = cfg["muestra_audio"]
assert set(DOCS_EVAL_IDS) <= set(ocr_crudo["id"]), "Algún documento de evaluación no está en el OCR de la Parte 1"

inv = inventario.set_index("id")
print("Fijada el", cfg["fecha"], "con semilla", cfg["semilla"])
print("Documentos de evaluación:", DOCS_EVAL_IDS)
print("Clips de evaluación:     ", CLIPS_EVAL_IDS)
display(inv.loc[CLIPS_EVAL_IDS, ["fuente", "origen", "duracion_seg"]])
print(f"Audio de evaluación: {inv.loc[CLIPS_EVAL_IDS, 'duracion_seg'].sum() / 60:.1f} min")

Fijada el 2026-10-24 con semilla 2026
Documentos de evaluación: ['doc_008', 'doc_002', 'doc_007', 'doc_014', 'doc_015', 'doc_016', 'doc_024', 'doc_020', 'doc_025', 'doc_022']
Clips de evaluación:      ['clip_007', 'clip_002', 'clip_004', 'clip_021', 'clip_016', 'clip_020', 'clip_024', 'clip_009', 'clip_022', 'clip_018']


,fuente,origen,duracion_seg
id,,,
clip_007,Wikimedia Commons,aurora_coronado.ogg,18.7
clip_002,Wikimedia Commons,agustino_1.ogg,20.5
clip_004,Wikimedia Commons,agustino_2.ogg,69.7
clip_021,OpenSLR 73,hablante pef_00610,79.2
clip_016,OpenSLR 73,hablante pem_02121,76.6
clip_020,OpenSLR 73,hablante pem_02484,77.3
clip_024,OpenSLR 73,hablante pem_07508,79.7
clip_009,OpenSLR 73,hablante pef_02121,78.0
clip_022,OpenSLR 73,hablante pem_04310,77.2


Audio de evaluación: 10.9 min


In [6]:
# Crea las plantillas de referencia (solo si no existen; nunca sobrescribe lo que ya escribí a mano)
for d in DOCS_EVAL_IDS:
    p = REF / f"{d}.txt"
    if not p.exists():
        p.write_text("", encoding="utf-8")
for c in CLIPS_EVAL_IDS:
    p = REF / f"{c}.txt"
    if not p.exists():
        p.write_text(inv.loc[c, "texto_oficial"] if inv.loc[c, "fuente"] == "OpenSLR 73" else "", encoding="utf-8")

def leer_ref(i):
    return (REF / f"{i}.txt").read_text(encoding="utf-8").strip()

vacias = [i for i in DOCS_EVAL_IDS + CLIPS_EVAL_IDS if not leer_ref(i)]
if vacias:
    print("Faltan referencias manuales. Abre y completa en VS Code estos archivos de datos/raw/referencias/:")
    for i in vacias:
        print("  ", f"{i}.txt")
    raise SystemExit("Completa las referencias y vuelve a ejecutar esta celda.")
print("Las 20 referencias están completas.")

Las 20 referencias están completas.
